In [2]:
"""
Fetches macro indicator data from FRED API and converts
it into text documents that LlamaIndex can embed and search.

Why we convert numbers to sentences:
    FRED returns raw floats - 5.33
    Embeddings on "5.33" are meaningless.
    "In 2024-01, the federal funds rate was 5.33% embeds correctly
    because the model understands what the number means in context
"""

'\nFetches macro indicator data from FRED API and converts\nit into text documents that LlamaIndex can embed and search.\n\nWhy we convert numbers to sentences:\n    FRED returns raw floats - 5.33\n    Embeddings on "5.33" are meaningless.\n    "In 2024-01, the federal funds rate was 5.33% embeds correctly\n    because the model understands what the number means in context\n'

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

# Quick check — confirms keys loaded, doesn't print actual values
print("OpenAI:", "OK" if os.getenv("OPENAI_API_KEY") else "MISSING")
print("FRED:", "OK" if os.getenv("FRED_API_KEY") else "MISSING")
print("NewsAPI:", "OK" if os.getenv("NEWS_API_KEY") else "MISSING")
print("NYT:", "OK" if os.getenv("NYTIMES_API_KEY") else "MISSING")

OpenAI: OK
FRED: OK
NewsAPI: OK
NYT: OK


In [4]:
from fredapi import Fred

fred = Fred(api_key=os.getenv("FRED_API_KEY"))

# Fetch the Federal Funds Rate — last 6 months
series = fred.get_series("FEDFUNDS")
last_6 = series.tail(6)

print("=== RAW from FRED (what we get) ===")
print(last_6)

=== RAW from FRED (what we get) ===
2025-12-01    3.72
2026-01-01    3.64
2026-02-01    3.64
2026-03-01    3.64
2026-04-01    3.64
2026-05-01    3.63
dtype: float64


In [5]:
# === CONVERTED to prose (what LlamaIndex will embed) ===
print("=== PROSE VERSION (what we embed) ===\n")

for date, value in last_6.items():
    date_str = date.strftime("%Y-%m")
    print(f"In {date_str}, the Federal Funds Rate was {value:.2f}%.")

=== PROSE VERSION (what we embed) ===

In 2025-12, the Federal Funds Rate was 3.72%.
In 2026-01, the Federal Funds Rate was 3.64%.
In 2026-02, the Federal Funds Rate was 3.64%.
In 2026-03, the Federal Funds Rate was 3.64%.
In 2026-04, the Federal Funds Rate was 3.64%.
In 2026-05, the Federal Funds Rate was 3.63%.


In [6]:
# All FRED series we want to index
FRED_SERIES = {
    "FEDFUNDS": "Federal Funds Rate",
    "CPIAUCSL": "Consumer Price Index (CPI)",
    "GDP":      "US Gross Domestic Product",
    "UNRATE":   "Unemployment Rate",
    "DGS10":    "10-Year Treasury Yield",
    "T10YIE":   "10-Year Breakeven Inflation Rate",
}

for series_id, series_name in FRED_SERIES.items():
    series = fred.get_series(series_id).tail(6)
    print(f"\n--- {series_name} ---")
    for date, value in series.dropna().items():
        print(f"In {date.strftime('%Y-%m')}, the {series_name} was {value:.2f}.")


--- Federal Funds Rate ---
In 2025-12, the Federal Funds Rate was 3.72.
In 2026-01, the Federal Funds Rate was 3.64.
In 2026-02, the Federal Funds Rate was 3.64.
In 2026-03, the Federal Funds Rate was 3.64.
In 2026-04, the Federal Funds Rate was 3.64.
In 2026-05, the Federal Funds Rate was 3.63.

--- Consumer Price Index (CPI) ---
In 2025-12, the Consumer Price Index (CPI) was 326.03.
In 2026-01, the Consumer Price Index (CPI) was 326.59.
In 2026-02, the Consumer Price Index (CPI) was 327.46.
In 2026-03, the Consumer Price Index (CPI) was 330.29.
In 2026-04, the Consumer Price Index (CPI) was 332.41.
In 2026-05, the Consumer Price Index (CPI) was 333.98.

--- US Gross Domestic Product ---
In 2024-10, the US Gross Domestic Product was 29825.18.
In 2025-01, the US Gross Domestic Product was 30042.11.
In 2025-04, the US Gross Domestic Product was 30485.73.
In 2025-07, the US Gross Domestic Product was 31098.03.
In 2025-10, the US Gross Domestic Product was 31422.53.
In 2026-01, the US Gr

In [7]:
from llama_index.core import Document

fred_documents = []

for series_id, series_name in FRED_SERIES.items():
    series = fred.get_series(series_id).tail(24)
    
    lines = [f"FRED Data: {series_name}\n"]
    for date, value in series.dropna().items():
        lines.append(f"In {date.strftime('%Y-%m')}, the {series_name} was {value:.2f}.")
    
    text = "\n".join(lines)
    
    fred_documents.append(Document(
        text=text,
        metadata={"source": "FRED", "series": series_id, "domain": "macro"}
    ))

print(f"Created {len(fred_documents)} FRED documents")
for doc in fred_documents:
    print(f"  - {doc.metadata['series']}: {len(doc.text)} chars")

Created 6 FRED documents
  - FEDFUNDS: 1110 chars
  - CPIAUCSL: 1303 chars
  - GDP: 1381 chars
  - UNRATE: 1041 chars
  - DGS10: 1161 chars
  - T10YIE: 1401 chars


These 6 cover the core macro signals that drive most finance/tech causal chains:

FEDFUNDS → interest rate decisions
CPIAUCSL → inflation
GDP → economic growth
UNRATE → labor market
DGS10 → bond market / risk sentiment
T10YIE → inflation expectations

In [9]:
import wikipediaapi
from llama_index.core import Document

wiki = wikipediaapi.Wikipedia(
    user_agent="ContextCore/1.0",
    language="en"
)

WIKI_TOPICS = [
    "Federal Reserve",
    "Interest rate",
    "Inflation",
    "Yield curve",
    "Recession",
    "Quantitative easing",
    "Semiconductor industry",
    "Stock market",
    "Gross domestic product",
]

wiki_documents = []

for topic in WIKI_TOPICS:
    page = wiki.page(topic)
    
    if not page.exists():
        print(f"  SKIP: {topic} not found")
        continue
    
    wiki_documents.append(Document(
        text=f"Wikipedia: {page.title}\n\n{page.text[:3000]}",
        metadata={"source": "Wikipedia", "title": page.title, "domain": "general"}
    ))
    
    print(f"  OK: {page.title} ({len(page.text[:3000])} chars)")

print(f"\nTotal wiki documents: {len(wiki_documents)}")

  OK: Federal Reserve (3000 chars)
  OK: Interest rate (3000 chars)
  OK: Inflation (3000 chars)
  OK: Yield curve (3000 chars)
  OK: Recession (3000 chars)
  OK: Quantitative easing (3000 chars)
  OK: Semiconductor industry (3000 chars)
  OK: Stock market (3000 chars)
  OK: Gross domestic product (3000 chars)

Total wiki documents: 9


In [10]:
import requests

NEWS_API_KEY = os.getenv("NEWS_API_KEY")

url = "https://newsapi.org/v2/everything"
params = {
    "q": "Federal Reserve interest rate",
    "language": "en",
    "sortBy": "relevancy",
    "pageSize": 5,
    "apiKey": NEWS_API_KEY
}

response = requests.get(url, params=params)
data = response.json()

print("Status:", data["status"])
print("Total results:", data["totalResults"])
print("\nFirst article:")
print("Title:", data["articles"][0]["title"])
print("Source:", data["articles"][0]["source"]["name"])
print("Description:", data["articles"][0]["description"])

Status: ok
Total results: 1498

First article:
Title: Inflation surged in April to the highest rate since 2023
Source: Business Insider
Description: The new CPI report showed inflation in April surged to the highest rate since 2023.


In [11]:
news_documents = []

for article in data["articles"]:
    title       = article.get("title") or ""
    description = article.get("description") or ""
    content     = article.get("content") or ""
    
    # Skip if too short to be useful
    if len(title + description) < 50:
        continue
    
    text = f"News Article: {title}\n"
    text += f"Source: {article['source']['name']}\n"
    text += f"Published: {article.get('publishedAt', '')[:10]}\n\n"
    text += f"{description}\n\n{content}"
    
    news_documents.append(Document(
        text=text,
        metadata={
            "source": "NewsAPI",
            "title": title,
            "url": article.get("url", ""),
            "domain": "finance"
        }
    ))

print(f"Created {len(news_documents)} news documents")
print(f"\nSample:\n{news_documents[0].text[:300]}")

Created 5 news documents

Sample:
News Article: Inflation surged in April to the highest rate since 2023
Source: Business Insider
Published: 2026-05-12

The new CPI report showed inflation in April surged to the highest rate since 2023.

Patrick T. Fallon / AFP via Getty Images
<ul><li>The new CPI report showed annual inflation spe


In [12]:
all_documents = fred_documents + wiki_documents + news_documents

print(f"Total documents ready to index: {len(all_documents)}")
print(f"\nBreakdown:")
print(f"  FRED:      {len(fred_documents)}")
print(f"  Wikipedia: {len(wiki_documents)}")
print(f"  NewsAPI:   {len(news_documents)}")

Total documents ready to index: 20

Breakdown:
  FRED:      6
  Wikipedia: 9
  NewsAPI:   5


In [13]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding

# Tell LlamaIndex which embedding model to use
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Set up ChromaDB — persists to disk in a folder called chroma_db
chroma_client     = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("contextcore")
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)

# This is the magic line — chunk → embed → store
print("Building index... (takes 30-60 seconds)")
index = VectorStoreIndex.from_documents(
    all_documents,
    storage_context=storage_context,
    show_progress=True,
)

print("\nDone! Index saved to ./chroma_db/")

Building index... (takes 30-60 seconds)


c:\Users\aarus\Context_core\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating embeddings: 100%|██████████| 20/20 [00:05<00:00,  3.36it/s]


Done! Index saved to ./chroma_db/


####adding more topics

In [1]:
import wikipediaapi
from llama_index.core import Document

wiki = wikipediaapi.Wikipedia(user_agent="ContextCore/1.0", language="en")

NEW_WIKI_TOPICS = [
    "Venture capital",
    "Startup company",
    "Artificial intelligence",
    "Cryptocurrency",
    "European Central Bank",
    "Bond market",
    "Foreign exchange market",
    "Mergers and acquisitions",
    "Initial public offering",
    "Supply chain",
    "Reserve Bank of India",
    "Tariff",
    "Trade war",
]

new_wiki_documents = []

for topic in NEW_WIKI_TOPICS:
    page = wiki.page(topic)
    if not page.exists():
        print(f"  SKIP: {topic} not found")
        continue
    new_wiki_documents.append(Document(
        text=f"Wikipedia: {page.title}\n\n{page.text[:3000]}",
        metadata={"source": "Wikipedia", "title": page.title, "domain": "general"}
    ))
    print(f"  OK: {page.title} ({len(page.text[:3000])} chars)")

print(f"\nTotal new wiki documents: {len(new_wiki_documents)}")

  OK: Venture capital (3000 chars)
  OK: Startup company (3000 chars)
  OK: Artificial intelligence (3000 chars)
  OK: Cryptocurrency (3000 chars)
  OK: European Central Bank (3000 chars)
  OK: Bond market (3000 chars)
  OK: Foreign exchange market (3000 chars)
  OK: Mergers and acquisitions (3000 chars)
  OK: Initial public offering (3000 chars)
  OK: Supply chain (3000 chars)
  OK: Reserve Bank of India (3000 chars)
  OK: Tariff (3000 chars)
  OK: Trade war (3000 chars)

Total new wiki documents: 13


In [2]:
import requests
import os

NEWS_QUERIES = [
    "venture capital AI investment",
    "stock market today",
    "tech industry news",
    "global economy",
    "startup funding",
    "Reserve Bank of India",
]

new_news_documents = []
seen_urls = set()

for query in NEWS_QUERIES:
    response = requests.get(
        "https://newsapi.org/v2/everything",
        params={
            "q": query,
            "language": "en",
            "sortBy": "relevancy",
            "pageSize": 10,
            "apiKey": os.getenv("NEWS_API_KEY")
        }
    )
    data = response.json()
    articles = data.get("articles", [])

    count = 0
    for a in articles:
        url = a.get("url", "")
        title = a.get("title") or ""
        description = a.get("description") or ""

        if url in seen_urls or len(title + description) < 50:
            continue
        seen_urls.add(url)

        text = f"News Article: {title}\n"
        text += f"Source: {a.get('source', {}).get('name', 'Unknown')}\n"
        text += f"Published: {a.get('publishedAt', '')[:10]}\n\n"
        text += f"{description}\n\n{a.get('content', '')}"

        new_news_documents.append(Document(
            text=text,
            metadata={
                "source": "NewsAPI",
                "title": title,
                "url": url,
                "published_at": a.get("publishedAt", "")[:10],
                "outlet": a.get("source", {}).get("name", ""),
                "domain": "general",
            }
        ))
        count += 1

    print(f"  '{query}': +{count} new articles")

print(f"\nTotal new news documents: {len(new_news_documents)}")

  'venture capital AI investment': +10 new articles
  'stock market today': +10 new articles
  'tech industry news': +10 new articles
  'global economy': +10 new articles
  'startup funding': +8 new articles
  'Reserve Bank of India': +10 new articles

Total new news documents: 58


In [3]:
import requests
import os
from datetime import datetime, timedelta

NYT_QUERIES = [
    "venture capital artificial intelligence",
    "Federal Reserve",
    "stock market",
    "startup technology",
]

new_nyt_documents = []
seen_nyt_urls = set()

from_date = (datetime.today() - timedelta(days=60)).strftime("%Y%m%d")
to_date   = datetime.today().strftime("%Y%m%d")

for query in NYT_QUERIES:
    response = requests.get(
        "https://api.nytimes.com/svc/search/v2/articlesearch.json",
        params={
            "q": query,
            "begin_date": from_date,
            "end_date": to_date,
            "sort": "relevance",
            "api-key": os.getenv("NYTIMES_API_KEY")
        }
    )
    data = response.json()
    articles = data.get("response", {}).get("docs", [])[:10]

    count = 0
    for a in articles:
        title    = a.get("headline", {}).get("main", "")
        abstract = a.get("abstract") or a.get("snippet") or ""
        lead     = a.get("lead_paragraph") or ""
        url      = a.get("web_url", "")
        pub_date = a.get("pub_date", "")[:10]

        if url in seen_nyt_urls or len(title + abstract) < 50:
            continue
        seen_nyt_urls.add(url)

        text = f"NYT Article: {title}\n\n"
        text += f"Published: {pub_date}\nSection: {a.get('section_name', '')}\n\n"
        text += f"{abstract}\n\n{lead}"

        new_nyt_documents.append(Document(
            text=text,
            metadata={
                "source": "NYT",
                "title": title,
                "url": url,
                "published_at": pub_date,
                "outlet": "New York Times",
                "domain": "general",
            }
        ))
        count += 1

    print(f"  '{query}': +{count} new articles")

print(f"\nTotal new NYT documents: {len(new_nyt_documents)}")

  'venture capital artificial intelligence': +10 new articles
  'Federal Reserve': +9 new articles
  'stock market': +9 new articles
  'startup technology': +1 new articles

Total new NYT documents: 29


In [4]:
import chromadb
from llama_index.core import VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

chroma_client     = chromadb.PersistentClient(path="c:/Users/aarus/Context_core/utils/chroma_db")
chroma_collection = chroma_client.get_or_create_collection("contextcore")
vector_store      = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context   = StorageContext.from_defaults(vector_store=vector_store)

print("Collection count BEFORE:", chroma_collection.count())

all_new_documents = new_wiki_documents + new_news_documents + new_nyt_documents

index = VectorStoreIndex.from_vector_store(vector_store, storage_context=storage_context)

print(f"\nInserting {len(all_new_documents)} new documents...")
for doc in all_new_documents:
    index.insert(doc)

print("\nCollection count AFTER:", chroma_collection.count())

Collection count BEFORE: 20

Inserting 100 new documents...

Collection count AFTER: 120
